In [1]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Common imports
import numpy as np
import pandas as pd
import os
from pathlib import Path

ROOT = Path.cwd().parent   # vì notebook nằm trong folder notebook/
DATA_DIR = ROOT / "dataset"

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Where to save the figures
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "datathon_image"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần
mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

A) 30 ngày 

B) 90 ngày 

C) 180 ngày  

D) 365 ngày 

In [2]:
# Orders.csv analyst
df_orders = pd.read_csv(DATA_DIR/'orders.csv')

In [3]:
df_orders.columns

Index(['order_id', 'order_date', 'customer_id', 'zip', 'order_status',
       'payment_method', 'device_type', 'order_source'],
      dtype='object')

In [4]:
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])
df_orders.sort_values(['customer_id','order_date'],inplace=True)

In [5]:
# Find the distance between 2 orders of customers
df_orders['inter_order_gap'] = df_orders.groupby('customer_id')['order_date'].diff().dt.days

median_gap = df_orders['inter_order_gap'].median()
median_gap

np.float64(144.0)

Q2 Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price − cogs)/price?

A) Premium

B) Performance

C) Activewear

D) Standard

In [6]:
df_products = pd.read_csv(DATA_DIR/'products.csv')
df_products.columns

Index(['product_id', 'product_name', 'category', 'segment', 'size', 'color',
       'price', 'cogs'],
      dtype='object')

In [7]:
df_products.isna().any()

product_id      False
product_name    False
category        False
segment         False
size            False
color           False
price           False
cogs            False
dtype: bool

In [8]:
df_products['gross_margin'] = (df_products['price'] - df_products['cogs'])/df_products['price']

In [9]:
avg_margin_by_segment = df_products.groupby('segment')['gross_margin'].mean()
best_segment = avg_margin_by_segment.idxmax()

best_segment

'Standard'

Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

A) defective

B) wrong_size

C) changed_mind

D) not_as_described

In [10]:
df_returns = pd.read_csv(DATA_DIR/'returns.csv')
df_returns.columns

Index(['return_id', 'order_id', 'product_id', 'return_date', 'return_reason',
       'return_quantity', 'refund_amount'],
      dtype='object')

In [11]:
df_returns.isna().any()

return_id          False
order_id           False
product_id         False
return_date        False
return_reason      False
return_quantity    False
refund_amount      False
dtype: bool

In [12]:
df_merged_prre = pd.merge(df_products,df_returns,on=['product_id'],how='inner')
df_streetwear = df_merged_prre[df_merged_prre['category'] == 'Streetwear'] 

reason_cnt = df_streetwear['return_reason'].value_counts()

In [13]:
reason_cnt.idxmax()

'wrong_size'

Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ chuyển đổi trung bình (conversion_rate) caonhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

A) organic_search

B) paid_search

C) email_campaign

D) social_media

In [14]:
orders = pd.read_csv(DATA_DIR / "orders.csv")
web = pd.read_csv(DATA_DIR / "web_traffic.csv")

# Ép kiểu ngày cho chắc chắn
orders["order_date"] = pd.to_datetime(orders["order_date"])
web["date"] = pd.to_datetime(web["date"])

# Chuẩn hóa về date, bỏ phần giờ nếu có
orders["date"] = orders["order_date"].dt.normalize()
web["date"] = web["date"].dt.normalize()

# Đếm số đơn hàng theo ngày và order_source
daily_orders = (
    orders
    .groupby(["date", "order_source"])
    .size()
    .reset_index(name="orders")
    .rename(columns={"order_source": "traffic_source"})
)

# Tính tổng sessions theo ngày và traffic_source
daily_sessions = (
    web
    .groupby(["date", "traffic_source"], as_index=False)["sessions"]
    .sum()
)

# Đảm bảo traffic_source cùng kiểu string
daily_orders["traffic_source"] = daily_orders["traffic_source"].astype(str)
daily_sessions["traffic_source"] = daily_sessions["traffic_source"].astype(str)

# Merge
q4 = daily_sessions.merge(
    daily_orders,
    on=["date", "traffic_source"],
    how="left"
)

q4["orders"] = q4["orders"].fillna(0)

# Conversion rate = orders / sessions
q4["conversion_rate"] = q4["orders"] / q4["sessions"]

# Trung bình conversion_rate theo từng traffic_source
result = (
    q4
    .groupby("traffic_source")["conversion_rate"]
    .mean()
    .sort_values(ascending=False)
)

print(result)
print("\nQ4 answer:", result.index[0])

traffic_source
organic_search    0.002055
paid_search       0.001624
social_media      0.001505
email_campaign    0.000867
referral          0.000711
direct            0.000585
Name: conversion_rate, dtype: float64

Q4 answer: organic_search


Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?

A) 12%

B) 25%

C) 39%

D) 54%

In [15]:
df_order_items = pd.read_csv(DATA_DIR/'order_items.csv')
df_order_items.columns

C:\Users\Admin\AppData\Local\Temp\ipykernel_20388\88907976.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_order_items = pd.read_csv(DATA_DIR/'order_items.csv')


Index(['order_id', 'product_id', 'quantity', 'unit_price', 'discount_amount',
       'promo_id', 'promo_id_2'],
      dtype='object')

In [16]:
df_order_items.isna().sum()

order_id                0
product_id              0
quantity                0
unit_price              0
discount_amount         0
promo_id           438353
promo_id_2         714463
dtype: int64

In [17]:
df_order_items['promo_id'].notnull().mean() * 100

np.float64(38.663493169565214)

Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
nhóm)

A) 55+

B) 25–34

C) 35–44

D) 45–54

In [18]:
df_customers = pd.read_csv(DATA_DIR/'customers.csv')
df_customers.columns

Index(['customer_id', 'zip', 'city', 'signup_date', 'gender', 'age_group',
       'acquisition_channel'],
      dtype='object')

In [19]:
df_customers.isna().any()

customer_id            False
zip                    False
city                   False
signup_date            False
gender                 False
age_group              False
acquisition_channel    False
dtype: bool

In [20]:
order_count = df_orders.groupby('customer_id').size().reset_index(name='order_count')

In [21]:
df_customers_extend = pd.merge(df_customers[df_customers['age_group'].notnull()],order_count,on='customer_id',how='left')
df_customers_extend['order_count'] = df_customers_extend['order_count'].fillna(0)

In [22]:
avg_orders_by_age = df_customers_extend.groupby('age_group')['order_count'].mean()
avg_orders_by_age.idxmax()

'55+'

Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
sales_train.csv?

A) West

B) Central

C) East

D) Cả ba vùng có doanh thu xấp xỉ bằng nhau

In [23]:
df_geo = pd.read_csv(DATA_DIR/'geography.csv')
df_geo.columns

Index(['zip', 'city', 'region', 'district'], dtype='object')

In [24]:
df_order_items['revenue'] = df_order_items['quantity'] * df_order_items['unit_price']
order_revenue = df_order_items.groupby('order_id')['revenue'].sum().reset_index()

In [25]:
df_merged = pd.merge(df_orders, order_revenue, on='order_id', how='inner')
df_merged = pd.merge(df_merged, df_geo, on='zip', how='inner')

In [26]:
revenue_by_region = df_merged.groupby('region')['revenue'].sum()
revenue_by_region.idxmax()

'East'

Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
thanh toán nào được sử dụng nhiều nhất?

A) credit_card

B) cod

C) paypal

D) bank_transfer

In [27]:
df_order_cancelled = df_orders[df_orders['order_status']=='cancelled']

payment_count = df_order_cancelled['payment_method'].value_counts()
payment_count.idxmax()

'credit_card'

Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
với products theo product_id)?

A) S

B) M

C) L

D) XL

In [28]:
return_cnt = df_merged_prre['size'].value_counts()

df_merged_orpr = pd.merge(df_order_items,df_products,on='product_id',how='inner')
order_cnt = df_merged_orpr['size'].value_counts()

In [29]:
df_rates = pd.DataFrame({'returns': return_cnt, 'orders': order_cnt})
df_rates['return_rate'] = df_rates['returns'] / df_rates['orders']

In [30]:
df_rates['return_rate'].idxmax()

'S'

Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
mỗi đơn hàng cao nhất?

A) 1 kỳ (trả một lần)

B) 3 kỳ

C) 6 kỳ

D) 12 kỳ

In [31]:
df_payments = pd.read_csv(DATA_DIR/'payments.csv')
df_payments.columns

Index(['order_id', 'payment_method', 'payment_value', 'installments'], dtype='object')

In [32]:
df_payments.isna().any()

order_id          False
payment_method    False
payment_value     False
installments      False
dtype: bool

In [33]:
avg_payment_by_installment = df_payments.groupby('installments')['payment_value'].mean()
avg_payment_by_installment.idxmax()

np.int64(6)